In [1]:
# vllm_local_r1.py
import os, re, json
from typing import List, Dict, Any
from vllm import LLM, SamplingParams
from tqdm import tqdm
tqdm.pandas()

/home/sdonev/data/conda/envs/vllm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 10-17 10:08:59 [__init__.py:216] Automatically detected platform cuda.


In [2]:
import pandas as pd

In [3]:
# 1) Point to your local model clone (copied from HF to disk beforehand)
# "/shares/animalwelfare.crs.uzh/llms/DeepSeek-R1-Distill-Qwen-32B"
# "/data/sdonev/llms/DeepSeek-R1-Distill-Qwen-7B"
MODEL_DIR = "/shares/animalwelfare.crs.uzh/llms/DeepSeek-R1-Distill-Qwen-32B"  # contains config.json, tokenizer.*, *.safetensors

# 2) Keep vLLM/HF offline at runtime
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"



In [34]:
llm = LLM(
    model=MODEL_DIR,
    tokenizer=MODEL_DIR,
    trust_remote_code=True,
    #dtype="bfloat16",         # or "float16"
    max_model_len=8192,
    tensor_parallel_size=1,
)



INFO 10-17 13:07:42 [utils.py:328] non-default args: {'tokenizer': '/shares/animalwelfare.crs.uzh/llms/DeepSeek-R1-Distill-Qwen-32B', 'trust_remote_code': True, 'max_model_len': 8192, 'disable_log_stats': True, 'model': '/shares/animalwelfare.crs.uzh/llms/DeepSeek-R1-Distill-Qwen-32B'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 10-17 13:07:43 [__init__.py:742] Resolved architecture: Qwen2ForCausalLM
INFO 10-17 13:07:43 [__init__.py:1815] Using max model len 8192
INFO 10-17 13:07:43 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=16384.
(EngineCore_DP0 pid=3718898) INFO 10-17 13:07:43 [core.py:654] Waiting for init message from front-end.
(EngineCore_DP0 pid=3718898) INFO 10-17 13:07:43 [core.py:76] Initializing a V1 LLM engine (v0.10.2) with config: model='/shares/animalwelfare.crs.uzh/llms/DeepSeek-R1-Distill-Qwen-32B', speculative_config=None, tokenizer='/shares/animalwelfare.crs.uzh/llms/DeepSeek-R1-Distill-Qwen-32B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_co

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Processed prompts:   0%|          | 0/1 [01:46<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


(EngineCore_DP0 pid=3718898) INFO 10-17 13:07:54 [parallel_state.py:1165] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=3718898) WARNING 10-17 13:07:54 [topk_topp_sampler.py:69] FlashInfer is not available. Falling back to the PyTorch-native implementation of top-p & top-k sampling. For the best performance, please install FlashInfer.


...

[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
(EngineCore_DP0 pid=3718898) INFO 10-17 13:07:54 [gpu_model_runner.py:2338] Starting to load model /shares/animalwelfare.crs.uzh/llms/DeepSeek-R1-Distill-Qwen-32B...
(EngineCore_DP0 pid=3718898) INFO 10-17 13:07:55 [gpu_model_runner.py:2370] Loading model from scratch...
(EngineCore_DP0 pid=3718898) INFO 10-17 13:07:55 [cuda.py:362] Using Flash Attention backend on V1 engine.


Loading safetensors checkpoint shards:   0% Completed | 0/8 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  12% Completed | 1/8 [00:33<03:55, 33.62s/it]
Loading safetensors checkpoint shards:  25% Completed | 2/8 [01:07<03:23, 33.86s/it]
Loading safetensors checkpoint shards:  38% Completed | 3/8 [01:41<02:49, 33.99s/it]
Loading safetensors checkpoint shards:  50% Completed | 4/8 [02:16<02:16, 34.25s/it]
Loading safetensors checkpoint shards:  62% Completed | 5/8 [02:50<01:42, 34.13s/it]
Loading safetensors checkpoint shards:  75% Completed | 6/8 [03:05<00:55, 27.82s/it]
Loading safetensors checkpoint shards:  88% Completed | 7/8 [03:40<00:30, 30.02s/it]
Loading safetensors checkpoint shards: 100% Completed | 8/8 [04:14<00:00, 31.43s/it]
Loading safetensors checkpoint shards: 100% Completed | 8/8 [04:14<00:00, 31.87s/it]
(EngineCore_DP0 pid=3718898) 


(EngineCore_DP0 pid=3718898) INFO 10-17 13:12:10 [default_loader.py:268] Loading weights took 255.18 seconds
(EngineCore_DP0 pid=3718898) INFO 10-17 13:12:11 [gpu_model_runner.py:2392] Model loading took 61.0609 GiB and 255.648798 seconds
(EngineCore_DP0 pid=3718898) INFO 10-17 13:12:29 [backends.py:539] Using cache directory: /home/sdonev/data/.cache/vllm/torch_compile_cache/785a0c439e/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=3718898) INFO 10-17 13:12:29 [backends.py:550] Dynamo bytecode transform time: 17.71 s
(EngineCore_DP0 pid=3718898) INFO 10-17 13:12:37 [backends.py:161] Directly load the compiled graph(s) for dynamic shape from the cache, took 6.548 s
(EngineCore_DP0 pid=3718898) INFO 10-17 13:12:38 [monitor.py:34] torch.compile takes 17.71 s in total
(EngineCore_DP0 pid=3718898) INFO 10-17 13:12:41 [gpu_worker.py:298] Available KV cache memory: 4.44 GiB
(EngineCore_DP0 pid=3718898) INFO 10-17 13:12:41 [kv_cache_utils.py:864] GPU KV cache size: 18,176 toke

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:05<00:00, 11.51it/s]


(EngineCore_DP0 pid=3718898) INFO 10-17 13:12:48 [gpu_model_runner.py:3118] Graph capturing finished in 7 secs, took 1.17 GiB
(EngineCore_DP0 pid=3718898) INFO 10-17 13:12:48 [gpu_worker.py:391] Free memory on device (78.61/79.21 GiB) on startup. Desired GPU memory utilization is (0.9, 71.29 GiB). Actual usage is 61.06 GiB for weight, 5.72 GiB for peak activation, 0.07 GiB for non-torch memory, and 1.17 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=3359052492` to fit into requested memory, or `--kv-cache-memory=11216995840` to fully utilize gpu memory. Current kv cache memory in use is 4768338636 bytes.
(EngineCore_DP0 pid=3718898) INFO 10-17 13:12:48 [core.py:218] init engine (profile, create kv cache, warmup model) took 36.67 seconds
INFO 10-17 13:12:49 [llm.py:295] Supported_tasks: ['generate']
INFO 10-17 13:12:49 [__init__.py:36] No IOProcessor plugins requested by the model
ERROR 10-17 18:00:43 [core_client.py:564] Engine core proc EngineC

In [5]:

SYSTEM = (
    "You are helping filter noisy NER disease recognition results."
)

SCHEMA_DISEASE = """
<json>
{
  "entities": [
    {
      "text": "<entity string>"
    }
  ]
}
</json>
""".strip()

ANIMAL_STUDIES_GUIDE_DISEASE = r"""

### OBJECTIVE ###
- IDENTIFY AND RETURN ONLY THE DISEASE/CONDITION(S) THAT THE STUDY INTENDS TO **IMPROVE/REVERSE/AMELIORATE** (THE THERAPEUTIC TARGET).
- RETURN OUTPUT STRICTLY USING **SCHEMA_DISEASE**.

### INPUT FORMAT ###
YOU WILL RECEIVE:
1) ABSTRACT: A SHORT TEXT DESCRIBING AN **ANIMAL** EXPERIMENT TESTING A NEW INTERVENTION.
2) ENTITY_LIST: A CANDIDATE LIST OF DISEASE-RELATED ENTITY ALREADY IDENTIFIED BY NER.  

### OUTPUT FORMAT (STRICT) ###
RETURN EXACTLY:
ANSWER: <json>
{
  "entities": [
    { "text": "<entity string>" }
  ]
}
</json>
- RETURN ZERO, ONE, OR MULTIPLE ENTITIES.
- **EACH "text" MUST BE COPIED VERBATIM FROM ENTITY_LIST** (DO NOT REWRITE, CANONICALIZE, OR INVENT).

### DECIDE (RULES) ###
For each entity in ENTITY_LIST, decide whether to keep it using the following rules:

1. Validity
   - Keep only complete disease or condition terms.
   - Keep abbreviations (e.g., "AD", "PD", "MS") as **separate entities** if they are used in the abstract to refer to valid diseases/conditions.
   - Do not keep:
     • Partial tokens (e.g., "chronic", "tactile")
     • Pure adjectives or modifiers (e.g., "ischemic", "parkinsonian", "inflammatory")
     • Overly unspecific concepts (e.g., "brain damage", "neuronal damage", "neurodegeneration")

2. Relevance
   - Keep the disease(s)/condition(s) that are the therapeutic target of the study.
   - Do not keep entities mentioned only as background, unrelated examples, or exclusion criteria.

3. Specificity
   - Keep both more general and more specific mentions of a disease when both appear in the study.
     • Example: If the text says “Alzheimer’s disease is a severe form of dementia… we tested a treatment for Alzheimer’s disease”, then both "Alzheimer’s disease" and "dementia" should be kept.
     • Example: If the text says “we treated seizures… more concretely slow-wave burst seizure which is a form of focal seizure”, then "seizure", "slow-wave burst seizure", and "focal seizure" should all be kept.

4. Composite/linked conditions
   - If the target is described as a consequence of another disease, keep both.
     • Example: If the study treats "post-stroke seizure", then annotate "seizure" and "stroke" separately.

5. Final decision
   - Keep the entity only if it satisfies:
     • Valid disease/condition term, AND
     • Relevant (main therapeutic target), AND
     • Not filtered out as adjective, unspecific or as a pure descriptor.

"""

FEW_SHOT_EXAMPLES_DISEASE = r"""
### FEW-SHOT EXAMPLES (ADAPTED TO SCHEMA) ###

Example 1
Abstract:  
Alzheimer’s disease is a severe form of dementia. In this study, we tested whether MKI-801 had a beneficial impact on memory impairment symptoms in Alzheimer’s disease mouse models.  

Entities: 
- Alzheimer’s disease  
- dementia  
- chronic  
- memory impairment  

Expected:
ANSWER: <json>
{
  "entities": [
    { "text": "Alzheimer’s disease" },
    { "text": "dementia" },
    { "text": "memory impairment" }
  ]
}
</json>

Example 2
Abstract:
"In this study, we used a mouse model to induce colitis with DSS to study the effect of probiotic Y. Crohn’s disease and ulcerative colitis are common forms of inflammatory bowel disease in humans."

Entities:
- colitis
- Crohn’s disease
- ulcerative colitis
- inflammatory bowel disease

Expected:
ANSWER: <json>
{
  "entities": [
    {
      "text": "colitis"
      
    }
  ]
}
</json>
""".strip()


In [6]:
SYSTEM_DRUG = (
    "You are helping filter noisy NER drug recognition results."
)


SCHEMA_DRUG = """
<json>
{
  "entities": [
    {
      "text": "<entity string>"
    }
  ]
}
</json>
""".strip()

ANIMAL_STUDIES_GUIDE_DRUG = r"""

### OBJECTIVE ###
- IDENTIFY AND RETURN ONLY THE DRUG/THERAPEUTIC INTERVENTION(S) THAT THE STUDY **ADMINISTERS/TESTS** (THE ACTIVE SUBSTANCE[S]).
- RETURN OUTPUT STRICTLY USING **SCHEMA_DRUG**.

### INPUT FORMAT ###
YOU WILL RECEIVE:
1) ABSTRACT: A SHORT TEXT DESCRIBING AN **ANIMAL** EXPERIMENT TESTING A NEW INTERVENTION.
2) ENTITY_LIST: A CANDIDATE LIST OF DRUG-RELATED ENTITIES ALREADY IDENTIFIED BY NER.  

### OUTPUT FORMAT (STRICT) ###
RETURN EXACTLY:
ANSWER: <json>
{
  "entities": [
    { "text": "<entity string>" }
  ]
}
</json>
- RETURN ZERO, ONE, OR MULTIPLE ENTITIES.
- **EACH "text" MUST BE COPIED VERBATIM FROM ENTITY_LIST** (DO NOT REWRITE, CANONICALIZE, OR INVENT).

### DECIDE (RULES) ###
For each entity in ENTITY_LIST, decide whether to keep it using the following rules:

1) Definition / What counts as a DRUG here
   - Keep **active pharmacological substances** that are **administered/tested** in the study (small molecules, peptides, biologics, experimental codes like "DFC4889", etc.).
   - **Include brand names and generic names** if either appears; keep them as separate entities if both appear.
   - **Include adjuvants/co-treatments** if they are administered (e.g., "ADY" when used with "DFC4889") — keep each as a **separate entity**.

2) Form and dosing details
   - **Keep only the substance name**, not the dose, route, schedule, or formulation.
     • Example: From "MKI-801 20 mg p.o." keep **"MKI-801"** only.
   - Exclude pure formulation words and vehicles (e.g., "saline", "vehicle", "corn oil", "CMC") **unless** they are pharmacologically active drugs under test.

3) Specificity and classes
   - Keep the **most specific** drug mention (e.g., "fingolimod", "ibuprofen").
   - **Also keep specific pharmacologic classes** when explicitly mentioned, such as:
     **"opioid receptor agonist"**, **"beta-blocker"**, **"SSRI"**, **"SNRI"**, **"tricyclic antidepressants"**, **"NSAID"**, **"NSAR"**.
   - **Do NOT keep** very generic lay categories lacking specificity, e.g., "pain killer", "analgesic", "antibiotic", "antidepressant", "blood thinner", "antioxidant".

4) Negation and controls
   - **Keep drug mentions even if negated** when referring to the intervention/control design.
     • Example: “the control rats did **not** receive **fingolimod**” → keep **"fingolimod"**.
   - Mentions of drugs only as unrelated background (not part of the tested/controlled interventions) should be excluded.

5) Conjugates, combinations, and linked forms
   - **Annotate conjugate/complex names as separate drug entities** when possible.
     • Example: “**ibuprofen glutathione conjugate**” → keep **"ibuprofen"** and **"glutathione"** (do not include the word "conjugate").
   - **Drug combinations** (e.g., “acetaminophen-codeine”) → keep **each component** as a separate entity when identifiable from ENTITY_LIST.

6) Idiomatic multiword drug phrases
   - If a **multiword phrase is idiomatic and specific** (established usage beyond the sum of words), **include modifiers that define its pharmacologic identity**.
     • Example: **"selective PPAR agonist"** → keep the full phrase **"selective PPAR agonist"**.
   - Do not include non-defining adjectives (“chronic”, “systemic”) unless they are part of an idiomatic, pharmacologically defining term.

7) Abbreviations and codes
   - **Keep experimental codes and abbreviations** (e.g., “DFC4889”, “MK-801”, “AZD1234”, “L-NAME”) if they denote administered substances in the study.
   - If an acronym clearly refers to a drug used in the experiment and appears in ENTITY_LIST, keep it as its own entity.

8) Exclusions (filter out)
   - Partial tokens (e.g., “selective” alone, “agonist” alone) unless part of an idiomatic drug/class term as in Rule 6.
   - Pure routes/schedules (e.g., “i.p.”, “p.o.”, “b.i.d.”), doses (“20 mg/kg”), and units.
   - Delivery devices and non-active carriers (e.g., “micelles”, “nanoparticle”, “liposome”) **unless** the payload drug name itself appears (then keep the payload drug).
   - Pure targets without drug identity (e.g., “PPAR” alone) unless the entity is an idiomatic drug class phrase per Rule 6.

9) Relevance
   - Keep drugs that are **administered** to animals in the experiment (treatment or comparator, including positive/standard controls).
   - Do not keep drugs mentioned **only** in unrelated background, prior literature, or as exclusion criteria.

10) Final decision checklist
   Keep the entity only if:
   - It denotes an **active substance** (or an allowed specific class term, Rule 3),
   - It is **relevant** to the interventions/control arms in the study,
   - It is **not** filtered out by the exclusions above.
"""

FEW_SHOT_EXAMPLES_DRUG = r"""
### FEW-SHOT EXAMPLES (ADAPTED TO SCHEMA) ###

Example 1  
Abstract:  
In this study, we tested the effects of MKI-801 20 mg p.o. on learning behavior in mice. MKI-801 is an NMDA receptor antagonist known to modulate cognitive processes.  

Entities:  
- MKI-801  
- NMDA receptor antagonist  
- 20 mg  
- p.o.  

Expected:  
ANSWER: <json>
{
  "entities": [
    { "text": "MKI-801" },
    { "text": "NMDA receptor antagonist" }
  ]
}
</json>

---

Example 2  
Abstract:  
The control rats did not receive fingolimod, whereas the treated group was given fingolimod (FTY720) daily for seven days.  

Entities:  
- fingolimod  
- FTY720  
- control rats  

Expected:  
ANSWER: <json>
{
  "entities": [
    { "text": "fingolimod" },
    { "text": "FTY720" }
  ]
}
</json>

---

Example 3  
Abstract:  
To study oxidative stress, we synthesized an ibuprofen glutathione conjugate and compared it with ibuprofen alone.  

Entities:  
- ibuprofen glutathione conjugate  
- ibuprofen  
- glutathione  
- oxidative stress  

Expected:  
ANSWER: <json>
{
  "entities": [
    { "text": "ibuprofen" },
    { "text": "glutathione" }
  ]
}
</json>
""".strip()


In [13]:
def build_prompt(abstract, entities, SYSTEM, ANIMAL_STUDIES_GUIDE, SCHEMA, FEW_SHOT_EXAMPLES):
    """
    Build a strict instruction prompt specialized for preclinical animal studies.
    - Preserves your schema and <json> wrapper.
    - Enforces ordering, no entity changes, ≤12-word evidence quotes.
    """
    ents = "\n".join(f"- {e['text']}" for e in entities)

    return (
        f"SYSTEM:\n{SYSTEM}\n\n"
        "USER:\n"
        f"{ANIMAL_STUDIES_GUIDE}\n\n"
        "### INPUT FORMAT ###\n"
        "- YOU WILL RECEIVE:\n"
        "  - A BIOMEDICAL ABSTRACT.\n"
        "  - A LIST OF DISEASE ENTITIES FOUND IN THAT ABSTRACT.\n\n"
        "### OUTPUT FORMAT ###\n"
        "- RETURN ONLY A JSON OBJECT INSIDE <json> AND </json> TAGS, MATCHING THIS SCHEMA:\n"
        f"{SCHEMA}\n\n"
        f"{FEW_SHOT_EXAMPLES}\n\n"
        f"NEW Abstract:\n<<<{abstract}>>>\n\n"
        "Entities:\n<<<\n"
        f"{ents}\n>>>\n"
        "Return your answer strictly as JSON inside <json>...</json>, nothing else:\n<<<\n"
        
    )

def build_prompt_llm_extractor(abstract, SYSTEM, ANIMAL_STUDIES_GUIDE, SCHEMA, FEW_SHOT_EXAMPLES):
    """
    Build a strict instruction prompt specialized for preclinical animal studies.
    - Designed for extracting disease/condition entities directly from the abstract (no entity list provided).
    - Preserves schema structure and <json> wrapper.
    - Enforces strict formatting and return constraints.
    """

    return (
        f"SYSTEM:\n{SYSTEM}\n\n"
        "USER:\n"
        f"{ANIMAL_STUDIES_GUIDE}\n\n"
        "### INPUT FORMAT ###\n"
        "- YOU WILL RECEIVE:\n"
        "  - A BIOMEDICAL ABSTRACT DESCRIBING AN ANIMAL STUDY.\n"
        "### OUTPUT FORMAT ###\n"
        "- RETURN ONLY A JSON OBJECT INSIDE <json> AND </json> TAGS, MATCHING THIS SCHEMA:\n"
        f"{SCHEMA}\n\n"
        f"{FEW_SHOT_EXAMPLES}\n\n"
        f"NEW Abstract:\n<<<{abstract}>>>\n\n"
        "Return your answer strictly as JSON inside <json>...</json>, nothing else:\n<<<\n"
    )

In [8]:
sampling = SamplingParams(
    temperature=0.0,
    top_p=1.0,
    max_tokens=2000,
    stop=["</json>"],   # <- important
)

def extract_json(text: str):
    # 1. Preferred case: inside <json>...</json>
    m = re.search(r"<json>\s*(\{.*\})\s*(?:</json>)?", text, flags=re.S)
    if m:
        #print(m.group(1))
        return json.loads(m.group(1))

    # 2. Fenced code block: ```json ... ```
    m = re.search(r"```json\s*(\{.*?\})\s*```", text, flags=re.S | re.I)
    if m:
        return json.loads(m.group(1))

    # 3. Any last JSON-looking object in the string
    return _parse_last_json_anywhere(text)


def _parse_last_json_anywhere(text: str):
    """
    Try to grab the last {...} block from the text and parse it as JSON.
    Useful if LLM prepends explanations or stray text.
    """
    matches = list(re.finditer(r"\{.*\}", text, flags=re.S))
    if not matches:
        raise ValueError(f"No JSON object found in: {text}")
    for m in reversed(matches):  # last one usually the valid one
        try:
            return json.loads(m.group())
        except json.JSONDecodeError:
            continue
    raise ValueError(f"Could not parse JSON from: {text}")

In [9]:
abstract = (
    "We conducted a randomized, double-blind trial evaluating adalimumab for "
    "moderate-to-severe rheumatoid arthritis. Participants received adalimumab "
    "or methotrexate. The primary endpoint was ACR20 at week 12. Background "
    "therapies such as NSAIDs were allowed. Past studies have also explored "
    "infliximab in similar populations."
)

# Instead of drugs, we focus on disease mentions:
entities = [
    {"text": "rheumatoid arthritis", "type": "DISEASE"},
    {"text": "ACR20", "type": "DISEASE"}  # maybe spurious, but included for demonstration
]

# Build a disease-specific prompt
prompt = build_prompt(
    abstract,
    entities,
    SYSTEM,
    ANIMAL_STUDIES_GUIDE_DISEASE,
    SCHEMA_DISEASE,
    FEW_SHOT_EXAMPLES_DISEASE
)

out = llm.generate([prompt], sampling)
text = out[0].outputs[0].text  # already stops before </json>
data = extract_json(text)
print(json.dumps(data, indent=2))

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.06s/it, est. speed input: 1031.11 toks/s, output: 31.19 toks/s]

{
  "entities": [
    {
      "text": "rheumatoid arthritis"
    }
  ]
}


In [11]:
abstract = ("We conducted a randomized, double-blind trial evaluating adalimumab for moderate-to-severe "
            "rheumatoid arthritis. Participants received adalimumab or methotrexate. The primary endpoint "
            "was ACR20 at week 12. Background therapies such as NSAIDs were allowed. Past studies have "
            "also explored infliximab in similar populations.")
entities = [
    {"text": "adalimumab", "type": "DRUG"},
    {"text": "methotrexate", "type": "DRUG"},
    {"text": "infliximab", "type": "DRUG"},
    {"text": "NSAIDs", "type": "DRUG"},
]

prompt = build_prompt(abstract, entities, SYSTEM_DRUG, ANIMAL_STUDIES_GUIDE_DRUG, SCHEMA_DRUG, FEW_SHOT_EXAMPLES_DRUG)
out = llm.generate([prompt], sampling)
text = out[0].outputs[0].text  # already stops before </json>
data = extract_json(text)
print(json.dumps(data, indent=2))

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.31s/it, est. speed input: 1418.07 toks/s, output: 31.34 toks/s]

{
  "entities": [
    {
      "text": "adalimumab"
    },
    {
      "text": "methotrexate"
    }
  ]
}


### LLM only

In [11]:
ANIMAL_STUDIES_GUIDE_DISEASE_LLM_ONLY = r"""

### OBJECTIVE ###
- IDENTIFY AND RETURN ONLY THE DISEASE/CONDITION(S) THAT THE STUDY INTENDS TO **IMPROVE/REVERSE/AMELIORATE** (THE THERAPEUTIC TARGET).
- RETURN OUTPUT STRICTLY USING **SCHEMA_DISEASE**.

### INPUT FORMAT ###
YOU WILL RECEIVE:
1) ABSTRACT: A SHORT TEXT DESCRIBING AN **ANIMAL** EXPERIMENT TESTING A NEW INTERVENTION.

### WHAT TO DO ###
From the ABSTRACT, **extract candidate disease/condition mentions directly from the text** (no pre-supplied list). Then apply the DECIDE rules below to keep only valid, relevant targets.

### OUTPUT FORMAT (STRICT) ###
RETURN EXACTLY:
ANSWER: <json>
{
  "entities": [
    { "text": "<entity string>" }
  ]
}
</json>
- RETURN ZERO, ONE, OR MULTIPLE ENTITIES.
- **EACH "text" MUST BE COPIED VERBATIM FROM ABSTRACT** (DO NOT REWRITE, CANONICALIZE, OR INVENT).

### DECIDE (RULES) ###
For each **candidate span you found in the abstract**, decide whether to keep it using the following rules:

1. Validity
   - Keep only complete disease or condition terms.
   - Keep abbreviations (e.g., "AD", "PD", "MS") as **separate entities** if they are used in the abstract to refer to valid diseases/conditions.
   - Do not keep:
     • Partial tokens (e.g., "chronic", "tactile")
     • Pure adjectives or modifiers (e.g., "ischemic", "parkinsonian", "inflammatory")
     • Overly unspecific concepts (e.g., "brain damage", "neuronal damage", "neurodegeneration")

2. Relevance
   - Keep the disease(s)/condition(s) that are the **therapeutic target** of the study (i.e., what the intervention aims to improve/reverse/ameliorate).
   - Do not keep entities mentioned only as background, unrelated examples, or exclusion criteria.

3. Specificity
   - Keep both more general and more specific mentions of a disease when both appear in the study.
     • Example: If the text says “Alzheimer’s disease is a severe form of dementia… we tested a treatment for Alzheimer’s disease”, then keep both "Alzheimer’s disease" and "dementia".
     • Example: If the text says “we treated seizures… more concretely slow-wave burst seizure which is a form of focal seizure”, then keep "seizure", "slow-wave burst seizure", and "focal seizure".

4. Composite/linked conditions
   - If the target is described as a consequence of another disease, keep both.
     • Example: If the study treats "post-stroke seizure", then annotate **both** "seizure" and "stroke" as separate entities.

5. Final decision
   - Keep the entity only if it satisfies:
     • Valid disease/condition term, AND
     • Relevant (main therapeutic target), AND
     • Not filtered out as adjective, unspecific, or a pure descriptor.
"""

FEW_SHOT_EXAMPLES_DISEASE_LLM_ONLY = r"""
### FEW-SHOT EXAMPLES (EXTRACTION FROM SCRATCH) ###

Example 1
Abstract:  
Alzheimer’s disease is a severe form of dementia. In this study, we tested whether MKI-801 had a beneficial impact on memory impairment symptoms in Alzheimer’s disease mouse models.  

→ The therapeutic targets described are "Alzheimer’s disease", "dementia", and "memory impairment".  
→ Words like "chronic" or adjectives are not present.  
→ All entities are copied verbatim from the abstract.

Expected:
ANSWER: <json>
{
  "entities": [
    { "text": "Alzheimer’s disease" },
    { "text": "dementia" },
    { "text": "memory impairment" }
  ]
}
</json>


Example 2
Abstract:
In this study, we used a mouse model in which colitis was induced by DSS to test the therapeutic effects of a probiotic compound. Crohn’s disease and ulcerative colitis are common human inflammatory bowel diseases.

→ The therapeutic target actually studied in the animal experiment is "colitis".  
→ "Crohn’s disease" and "ulcerative colitis" are mentioned only as background examples (not treated in this study).  
→ Therefore, only "colitis" is kept.

Expected:
ANSWER: <json>
{
  "entities": [
    { "text": "colitis" }
  ]
}
</json>


Example 3
Abstract:
We developed a novel compound that reduces neuronal loss and improves motor symptoms in a mouse model of Parkinson’s disease.

→ The disease being targeted is "Parkinson’s disease".  
→ Phrases like "neuronal loss" are descriptive, not valid disease entities.

Expected:
ANSWER: <json>
{
  "entities": [
    { "text": "Parkinson’s disease" }
  ]
}
</json>


Example 4
Abstract:
The compound was tested in a rodent model of post-stroke seizure to evaluate its potential to alleviate seizure severity after ischemic stroke.

→ The text describes a composite target: "post-stroke seizure".  
→ Per rules, both "seizure" and "stroke" are annotated separately.

Expected:
ANSWER: <json>
{
  "entities": [
    { "text": "seizure" },
    { "text": "stroke" }
  ]
}
</json>


Example 5
Abstract:
We tested an anti-inflammatory peptide in a rat model of multiple sclerosis (MS) to determine its effect on demyelination.

→ The target is "multiple sclerosis", which is also abbreviated as "MS" in the same abstract.  
→ Both should be kept as separate valid mentions.

Expected:
ANSWER: <json>
{
  "entities": [
    { "text": "multiple sclerosis" },
    { "text": "MS" }
  ]
}
</json>
""".strip()

## apply to PreclinIE test set

In [32]:
def apply_llm_cleanup(row, entities_col_name, entity_type, max_retries=3, llm_only=False):
    abstract = row['Text']
    entities_str = row[entities_col_name]

    # Handle NaN, None, or empty string
    if not entities_str or str(entities_str).strip() == "" or pd.isna(entities_str):
        return entities_str

    # Build fake entities from the pipe-separated string
    entities = [
        {"text": term.strip(), "type": entity_type}
        for term in str(entities_str).split("|") if term.strip()
    ]
    original_entities = {ent["text"] for ent in entities}
    if (len(original_entities) == 1) and (not llm_only):
        entity_txt = original_entities.pop()
        print(f"single entity, returning: {entity_txt}")
        return entity_txt
    

    # Build prompt
    if entity_type.upper() == 'DISEASE':
        if llm_only:
            print("Building disease prompt for LLM only extraction...")
            prompt = build_prompt_llm_extractor(
                abstract,
                SYSTEM, ANIMAL_STUDIES_GUIDE_DISEASE_LLM_ONLY, SCHEMA_DISEASE, FEW_SHOT_EXAMPLES_DISEASE_LLM_ONLY
            )
        else:
            print("Building disease prompt...")
            prompt = build_prompt(
                abstract, entities,
                SYSTEM, ANIMAL_STUDIES_GUIDE_DISEASE, SCHEMA_DISEASE, FEW_SHOT_EXAMPLES_DISEASE
            )
    else:
        print("Building drug prompt...")
        prompt = build_prompt(
            abstract, entities,
            SYSTEM_DRUG, ANIMAL_STUDIES_GUIDE_DRUG, SCHEMA_DRUG, FEW_SHOT_EXAMPLES_DRUG
        )

    # Try extraction with retries
    for attempt in range(1, max_retries + 1):
        try:
            out = llm.generate([prompt], sampling)
            text = out[0].outputs[0].text
            print(text)
            data = extract_json(text)  # may raise

            primary_targets = [
                ent["text"] for ent in data.get("entities", [])
                #if ent.get("role") == "primary_target"
            ]

            # ✅ Validation: all extracted must be in the original set
            if llm_only or all(pt in original_entities for pt in primary_targets):
                result = "|".join(primary_targets)
                print("original entities: ", original_entities)
                print(f"output: {result}")
                return result
            else:
                print(f"⚠️ attempt {attempt}: extracted entities not in original list → {primary_targets}")
                if attempt == max_retries:
                    print("❌ Giving up after max retries. Returning original entities.")
                    return entities_str
                continue

        except Exception as e:
            print(f"❌ attempt {attempt} failed: {e}")
            if attempt == max_retries:
                print("⚠️ Giving up after max retries. Returning original entities.")
                return entities_str

In [35]:
prompt_id = "prompt1_32B_FS_LLM_ONLY"
target_col = "conditions" # conditions, interventions
entity_type = "DISEASE"
llm_only_extractor = True

for batch_nr in range(10):
    batch_id = f"batch_{batch_nr}"
    output_path = f"./bert_ner_predictions/llm_cleaned/llm_{target_col}_{batch_id}_{prompt_id}.csv"

    # Skip if output already exists
    if os.path.exists(output_path):
        print(f"SKIPPING {batch_id} — output already exists.\n")
        continue

    print(f"PROCESSING {batch_id}\n")

    # Load input CSV
    input_path = f"./bert_ner_predictions/predictions_target_{batch_id}.csv"
    to_clean = pd.read_csv(input_path)

    # Choose column name
    if llm_only_extractor:
        new_col_name = f"unique_{target_col}_LLM_extractor_{prompt_id}"
    else:
        new_col_name = f"unique_{target_col}_biolinkbert_llm_clean_{prompt_id}"

    # Apply cleanup
    tqdm.pandas()
    to_clean[new_col_name] = to_clean.progress_apply(
        lambda row: apply_llm_cleanup(
            row,
            entities_col_name=f"unique_{target_col}_biolinkbert",
            entity_type=entity_type,
            llm_only=llm_only_extractor
        ),
        axis=1
    )

    # If LLM returned empty, fall back to BiolinkBERT entities
    mask = to_clean[new_col_name].str.strip() == ""
    to_clean.loc[mask, new_col_name] = to_clean.loc[mask, f"unique_{target_col}_biolinkbert"]

    # Save output
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    to_clean.to_csv(output_path, index=False)

    print(f"Saved cleaned file: {output_path}\n")

PROCESSING batch_0



  0%|          | 0/69 [00:00<?, ?it/s]

Building disease prompt for LLM only extraction...







Adding requests: 100%|██████████| 1/1 [00:00<00:00, 88.45it/s]A





Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]




Processed prompts: 100%|██████████| 1/1 [00:50<00:00, 50.07s/it, est. speed input: 37.63 toks/s, output: 39.95 toks/s]




Processed prompts: 100%|██████████| 1/1 [00:50<00:00, 50.07s/it, est. speed input: 37.63 toks/s, output: 39.95 toks/s]


Okay, so I need to help filter the noisy NER disease recognition results from the given abstract. The objective is to identify and return only the disease or condition that the study intends to improve, reverse, or ameliorate. I have to follow the DECIDE rules strictly.

First, I'll read through the abstract carefully to understand what the study is about. The abstract is about the ameliorative potential of a plant extract against chronic constriction injury-induced neuropathic pain in rats. The study investigates the potential of Alstonia scholaris in treating neuropathic pain caused by chronic constriction injury of the sciatic nerve.

Now, I need to extract candidate disease/condition mentions directly from the text. Let's go through the abstract sentence by sentence.

In the background, it mentions that the plant is used to treat various disorders like asthma, bronchitis, diarrhea, dysentery, and malaria. These are all valid disease terms, but I need to check if they are the therap

...

Alright, let's tackle this problem step by step. I need to extract the disease or condition that the study aims to improve or reverse. The abstract is about a study using a selective estrogen receptor modulator (SERM) called bazedoxifene (BZA) to enhance CNS remyelination in a rodent model. 

First, I'll read through the abstract to identify any disease mentions. The key terms I notice are "multiple sclerosis (MS)" and "focal demyelination." 

Now, applying the DECIDE rules:

1. **Validity**: Both "multiple sclerosis" and "MS" are valid disease terms. "Focal demyelination" is a specific condition related to MS, so it's also valid.

2. **Relevance**: The study's therapeutic target is the remyelination process in MS. While "focal demyelination" is mentioned in the context of the model used, it's a specific manifestation of MS. However, the main focus is on MS itself as the disease being treated.

3. **Specificity**: The study mentions both the general disease (MS) and a specific aspect (






Adding requests: 100%|██████████| 1/1 [00:00<00:00, 189.95it/s]





Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]




Processed prompts: 100%|██████████| 1/1 [00:11<00:00, 11.62s/it, est. speed input: 160.81 toks/s, output: 40.27 toks/s]




100%|██████████| 70/70 [14:28<00:00, 10.48s/it]

Alright, let's tackle this problem step by step. I'm supposed to extract the disease or condition that the study aims to improve or reverse. The abstract is about an animal study testing memantine's effects on tau phosphorylation induced by chronic restraint stress in mice.

First, I'll read through the abstract to identify any disease mentions. The abstract mentions "Alzheimer's disease-related tau hyperphosphorylation." So, "Alzheimer's disease" is a clear candidate. It's a valid disease term and is directly linked to the study's focus on tau phosphorylation.

Next, I'll check if there are any other disease mentions. The study uses a model of chronic restraint stress (CRS), which is a method to induce stress in mice. However, CRS itself isn't a disease but a condition used to simulate stress. The therapeutic target here is the effect on Alzheimer's disease-related tau phosphorylation, not CRS itself.

I also notice terms like "tau phosphorylation" and "glutamate neurotransmission abn






Adding requests: 100%|██████████| 1/1 [00:00<00:00, 210.96it/s]





Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]




Processed prompts: 100%|██████████| 1/1 [00:15<00:00, 15.52s/it, est. speed input: 113.09 toks/s, output: 40.28 toks/s]




100%|██████████| 70/70 [14:44<00:00, 12.63s/it]

Okay, so I need to help filter the noisy NER disease recognition results from the given abstract. The objective is to identify and return only the diseases or conditions that the study intends to improve, reverse, or ameliorate. I have to follow the DECIDE rules strictly.

First, I'll read through the abstract carefully to extract all candidate disease mentions. The abstract is about a study using cannabis-based medicine in a mouse model of Alzheimer's disease (AD). They mention several things like memory preservation, learning impairment, Abeta42 peptide levels, plaques, astrogliosis, microgliosis, inflammation, and some gene expressions related to thioredoxin 2 and Wnt16.

Now, applying the DECIDE rules:

1. **Validity**: I need to keep only complete disease terms. So, "Alzheimer's disease" and "AD" are valid. "Memory impairment" and "learning impairment" are specific conditions, so they should be kept. "Astrogliosis" and "microgliosis" are pathological conditions, so they are valid.

In [ ]:
#to_clean = pd.read_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_to_embeddings_ontologies/drug_disease_mapped_preclinical.csv")
batch_id = "batch_1"
to_clean = pd.read_csv(f"./bert_ner_predictions/predictions_target_{batch_id}.csv")
to_clean.head()

In [31]:
prompt_id = "prompt5_32B_FS"
new_col_name = f"unique_conditions_biolinkbert_llm_clean_{prompt_id}"
to_clean[new_col_name] = to_clean.progress_apply(
    lambda row: apply_llm_cleanup(row, entities_col_name="unique_conditions_biolinkbert", entity_type="DISEASE"),
    axis=1
)


  0%|          | 0/67 [00:00<?, ?it/s]

single entity, returning: neuropathic pain
single entity, returning: neuropathic pain
Building disease prompt...



Adding requests: 100%|██████████| 1/1 [00:00<00:00, 199.37it/s]

  6%|▌         | 4/67 [00:01<00:19,  3.28it/s]

ANSWER: <json>
{
  "entities": [
    { "text": "visceral hypersensitivity" },
    { "text": "gastric hyperalgesia" }
  ]
}

original entities:  {'gastric', 'gastric hyperalgesia', 'visceral hypersensitivity'}
output: visceral hypersensitivity|gastric hyperalgesia
Building disease prompt...



Adding requests: 100%|██████████| 1/1 [00:00<00:00, 198.34it/s]

  7%|▋         | 5/67 [00:02<00:38,  1.62it/s]

...

ANSWER: <json>
{
  "entities": [
    { "text": "ad" },
    { "text": "alzheimer's disease" }
  ]
}

original entities:  {'ad', "alzheimer's disease"}
output: ad|alzheimer's disease
Building disease prompt...



Adding requests: 100%|██████████| 1/1 [00:00<00:00, 213.11it/s]

100%|██████████| 67/67 [01:09<00:00,  1.61s/it]

ANSWER: <json>
{
  "entities": [
    {
      "text": "pain"
    },
    {
      "text": "sci pain"
    },
    {
      "text": "neuropathic spinal cord injury pain"
    },
    {
      "text": "neuropathic spinal cord injury"
    },
    {
      "text": "neuropathic sci pain"
    }
  ]
}

original entities:  {'neuropathic sci pain', 'neuropathic spinal cord injury pain', 'sci', 'neuropathic spinal cord injury', 'sci pain', 'pain'}
output: pain|sci pain|neuropathic spinal cord injury pain|neuropathic spinal cord injury|neuropathic sci pain
Building disease prompt...



Adding requests: 100%|██████████| 1/1 [00:00<00:00, 180.52it/s]

100%|██████████| 67/67 [01:10<00:00,  1.06s/it]

ANSWER: <json>
{
  "entities": [
    {
      "text": "sma"
    },
    {
      "text": "spinal muscular atrophy"
    }
  ]
}

original entities:  {'spinal muscular atrophy', 'sma'}
output: sma|spinal muscular atrophy


In [33]:
mask = to_clean[new_col_name].str.strip() == ""

to_clean.loc[mask, new_col_name] = (
    to_clean.loc[mask, "unique_conditions_biolinkbert"]
)

In [35]:
to_clean.to_csv(f"./bert_ner_predictions/llm_cleaned/llm_disease_{batch_id}_{prompt_id}.csv", index=False)

## Prep full dataset

In [85]:
to_clean = pd.read_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/mapped_preclinical_data_with_mondo_parents_mondo_cleaned.csv")[['PMID', 'disease_term_mondo_parent_clean']]
to_clean['PMID'] = to_clean['PMID'].astype(str)

In [86]:
to_clean.shape

(547365, 2)

In [87]:
doc_id = "25681574"
to_clean[to_clean['PMID']==doc_id]

,PMID,disease_term_mondo_parent_clean
416008,25681574,spinal cord demyelination|multiple sclerosis


In [88]:
target_pmids=list(to_clean.PMID)

In [89]:
target_pmids[:10]

['31733831',
 '31733833',
 '31733925',
 '31733940',
 '31734027',
 '31734152',
 '31734217',
 '31734231',
 '31734261',
 '31734269']

In [90]:
v1_file = "/shares/animalwelfare.crs.uzh/preclin_ner_full_ds/pubmed_animal_docs_large/drug_and_disease_content.csv" # => case when we had disease, but possibly missed
v2_file = "/shares/animalwelfare.crs.uzh/preclin_ner_full_ds/pubmed_animal_docs_large/drug_no_disease_content.csv" # => case when no disease was found at all

combined_text_df = pd.read_csv(v1_file)[['PMID','Text']]
combined_text_df_2 = pd.read_csv(v2_file)[['PMID','Text']]

/sctmp/sdonev/ipykernel_3505922/89608249.py:5: DtypeWarning: Columns (1,3) have mixed types. Specify dtype option on import or set low_memory=False.
  combined_text_df_2 = pd.read_csv(v2_file)[['PMID','Text']]


In [91]:
# Combine and drop duplicates if needed
combined_text_df = pd.concat([combined_text_df, combined_text_df_2], ignore_index=True)

# remove duplicates based on PMID
combined_text_df = combined_text_df.drop_duplicates(subset=["PMID"]).reset_index(drop=True)
combined_text_df['PMID'] = combined_text_df['PMID'].astype(str)
print(f"Combined rows: {len(combined_text_df)}")
combined_text_df.head()

Combined rows: 1592729


,PMID,Text
0,19118093,Trauma-hemorrhagic shock-induced pulmonary epi...
1,19118098,Role of central nervous system aldosterone syn...
2,19118113,Early but not late administration of glucagon-...
3,19118134,Synergy between enzyme inhibitors of fatty aci...
4,19118162,Nox4 NADPH oxidase mediates oxidative stress a...


In [92]:
to_clean = to_clean.merge(combined_text_df[['PMID','Text']], on="PMID", how="left")

In [93]:
num_nan = to_clean["Text"].isna().sum()
print(f"Rows where Text is NaN: {num_nan} out of {len(to_clean)}")

# Optional: see which PMIDs are missing text
missing_text_df = to_clean[to_clean["Text"].isna()]
missing_text_df.head()

Rows where Text is NaN: 1304 out of 547365


,PMID,disease_term_mondo_parent_clean,Text
546061,20661872,Parkinson disease|basal ganglia disorder,NaN
546062,7690856,multiple sclerosis,NaN
546063,9068857,Parkinson disease|basal ganglia disorder,NaN
546064,13981230,multiple sclerosis,NaN
546065,13981231,multiple sclerosis,NaN


In [55]:
to_clean

,PMID,disease_term_mondo_parent_clean,Text
0,31733831,asthma,Isorhynchophylline exerts anti-asthma effects ...
1,31733833,myocardial infarction,Exercise protects the heart against myocardial...
2,31733925,systemic lupus erythematosus,Therapeutic effects of soluble human leukocyte...
3,31733940,cognitive disorder,Resolving and Rescuing Developmental Miswiring...
4,31734027,oxaliplatin-induced peripheral neuropathy|cumu...,Improvement of peripheral vascular impairment ...
...,...,...,...
547360,39278849,multiple sclerosis,NaN
547361,2434820,Parkinson disease|basal ganglia disorder,NaN
547362,3300471,multiple sclerosis,NaN
547363,25052192,multiple sclerosis,NaN


In [94]:
# case-insensitive match
mask_ms = to_clean["disease_term_mondo_parent_clean"].str.contains("multiple sclerosis", case=False, na=False)

# split into two dataframes
df_ms = to_clean[mask_ms].copy()
df_other = to_clean[~mask_ms].copy()

# optional: sanity check
print(f"Rows with MS: {len(df_ms)}")
print(f"Rows without MS: {len(df_other)}")

Rows with MS: 6459
Rows without MS: 540906


In [107]:
save_chunks_dir = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/chunks_for_LLM_cleaning"

In [110]:
df_ms.to_csv(f"{save_chunks_dir}/chunk_0.csv")

In [111]:
import math

chunk_size = 15_000
num_chunks = math.ceil(len(df_other) / chunk_size)

print(f"Splitting {len(df_other)} rows into {num_chunks} chunks...")

for i in range(num_chunks):
    start = i * chunk_size
    end = start + chunk_size
    chunk = df_other.iloc[start:end]
    
    out_path = os.path.join(save_chunks_dir, f"chunk_{i+1}.csv")
    chunk.to_csv(out_path, index=False)
    
    print(f"Saved chunk {i+1}/{num_chunks}: {len(chunk)} rows → {out_path}")

print("✅ Done splitting and saving all chunks.")

Splitting 540906 rows into 37 chunks...
Saved chunk 1/37: 15000 rows → /shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/chunks_for_LLM_cleaning/chunk_1.csv
Saved chunk 2/37: 15000 rows → /shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/chunks_for_LLM_cleaning/chunk_2.csv
Saved chunk 3/37: 15000 rows → /shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/chunks_for_LLM_cleaning/chunk_3.csv
Saved chunk 4/37: 15000 rows → /shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/chunks_for_LLM_cleaning/chunk_4.csv
Saved chunk 5/37: 15000 rows → /shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/chunks_for_LLM_cleaning/chunk_5.csv
Saved chunk 6/37: 15000 rows → /shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/chunks_for_LLM_cleaning/chunk_6.csv
Saved chunk 7/37: 15000 rows → /shares/animalwelfare.c